# 🧠 Forward Pass
**Course:** Deep Learning  
**Lecture:** 2 — MLP, Backpropagation, and Activation Functions  
**Part:** a — Forward Pass

---

> **Where we are:** Before we can train a network to learn, we must understand how it computes its output from input data via layers of neurons.

**What you'll learn:**
- Layer-by-layer computation: how input propagates through hidden layers
- Matrix multiplication chain: shapes at every step
- Dense layer mechanics: $\mathbf{h} = f(\mathbf{W}\mathbf{x} + \mathbf{b})$
- Batch processing: how shapes change with the batch dimension

**Exam relevance:** ⭐⭐⭐ (Matrix shapes are CRITICAL for exam. You must be able to calculate weight shapes and output shapes.)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 📖 1. The Dense Layer
A dense (fully connected) layer applies a linear transformation followed by a non-linear activation function.

Given an input matrix $\mathbf{X}$ of shape `(batch_size, input_features)`, a weight matrix $\mathbf{W}$ of shape `(input_features, output_features)`, and a bias vector $\mathbf{b}$ of shape `(output_features,)`:
$$\mathbf{Z} = \mathbf{X} \mathbf{W} + \mathbf{b}$$
$$\mathbf{H} = f(\mathbf{Z})$$

Where $f$ is an activation function (like ReLU or Sigmoid).

## 🔀 Your Options: Implementing a Dense Layer
At this step, you can choose between implementing layers purely with NumPy, or using a framework like Keras.

| Option | Pros | Cons | When to use |
|---|---|---|---|
| **NumPy (From scratch)** | Deepest understanding of the math | Slow, no automatic differentiation | Educational purposes, learning the fundamentals |
| **Keras (`tf.keras.layers.Dense`)** | Highly optimized, handles gradients automatically | Masks the underlying mathematics | Production, complex architectures, practical work |

In [ ]:
def relu(z):
    return np.maximum(0, z)

# ✏️ YOUR TURN
# Implement a single dense layer function
def dense_layer_student(x, W, b, activation):
    """
    Computes a forward pass through a single dense layer.
    
    Args:
        x: Input array of shape (batch_size, input_dim)
        W: Weight matrix of shape (input_dim, output_dim)
        b: Bias vector of shape (output_dim,)
        activation: A function to apply element-wise to the output
        
    Returns:
        h: Output array of shape (batch_size, output_dim)
    """
    ### YOUR CODE HERE ###
    # Hint 1: Use np.dot or the @ operator for matrix multiplication
    # Hint 2: Add the bias vector (broadcasting handles batch size automatically!)
    # Hint 3: Apply the activation function
    
    # z = ...
    # h = ...
    # return h
    pass

# Test your code
x_test = np.ones((2, 3)) # batch_size=2, input_dim=3
W_test = np.ones((3, 4)) # input_dim=3, output_dim=4
b_test = np.zeros(4)

# Uncomment to test
# out = dense_layer_student(x_test, W_test, b_test, relu)
# print("Output shape:", out.shape) # Should be (2, 4)
# print("Output:", out) # Should be an array of 3s

In [ ]:
# ✅ SOLUTION
def dense_layer(x, W, b, activation):
    # Linear transformation: XW + b
    z = x @ W + b
    # Non-linear activation
    h = activation(z)
    return h

out = dense_layer(x_test, W_test, b_test, relu)
print("Solution Output Shape:", out.shape)

## 📖 2. Multi-Layer Perceptron (MLP)
An MLP simply chains these dense layers together. The output of one layer becomes the input to the next.
We will build a 3-layer MLP:
1. Hidden Layer 1: 8 neurons, ReLU
2. Hidden Layer 2: 4 neurons, ReLU
3. Output Layer: 1 neuron, Linear (no activation)

In [ ]:
# Let's set up the weights for a 3-layer MLP
batch_size = 32
input_dim = 2

X = np.random.randn(batch_size, input_dim)

# Layer 1: input_dim -> 8
W1 = np.random.randn(input_dim, 8)
b1 = np.random.randn(8)

# Layer 2: 8 -> 4
W2 = np.random.randn(8, 4)
b2 = np.random.randn(4)

# Layer 3: 4 -> 1
W3 = np.random.randn(4, 1)
b3 = np.random.randn(1)

# ✏️ YOUR TURN
# Compute the forward pass through all 3 layers using your dense_layer function
### YOUR CODE HERE ###
# h1 = ...
# h2 = ...
# out = ... # (use lambda x: x for linear activation)

# Print shapes at each step!
# print("Input shape:", X.shape)
# print("H1 shape:", h1.shape)
# print("H2 shape:", h2.shape)
# print("Output shape:", out.shape)

In [ ]:
# ✅ SOLUTION
h1 = dense_layer(X, W1, b1, relu)
print("H1 shape:", h1.shape)

h2 = dense_layer(h1, W2, b2, relu)
print("H2 shape:", h2.shape)

out_np = dense_layer(h2, W3, b3, lambda x: x)
print("Output shape:", out_np.shape)
print("\nFirst 3 outputs (NumPy):\n", out_np[:3])

## 🔬 3. Comparison with Keras
Let's build the exact same network in Keras, manually set its weights to match our NumPy arrays, and verify the outputs are identical.

In [ ]:
# Build Keras model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear')
])

# Set weights to match our NumPy implementation
model.layers[0].set_weights([W1, b1])
model.layers[1].set_weights([W2, b2])
model.layers[2].set_weights([W3, b3])

# Forward pass
out_keras = model.predict(X, verbose=0)

print("First 3 outputs (Keras):\n", out_keras[:3])

# Verify they match
np.testing.assert_allclose(out_np, out_keras, rtol=1e-5, atol=1e-5)
print("\n\u2705 Keras and NumPy outputs match perfectly!")

## 🔬 4. Shape Flow Visualization
The matrix multiplication rules determine the shapes in a neural network forward pass.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

boxes = [
    {"text": "Input\n(batch, 2)", "pos": 0.1},
    {"text": "Hidden 1\n(batch, 8)", "pos": 0.35},
    {"text": "Hidden 2\n(batch, 4)", "pos": 0.65},
    {"text": "Output\n(batch, 1)", "pos": 0.9}
]

for i, b in enumerate(boxes):
    ax.text(b['pos'], 0.5, b['text'], ha='center', va='center', 
            bbox=dict(facecolor='lightblue', boxstyle='round,pad=1', alpha=0.8), fontsize=12)
    
    if i < len(boxes) - 1:
        ax.annotate('', xy=(boxes[i+1]['pos']-0.08, 0.5), xytext=(b['pos']+0.08, 0.5),
                    arrowprops=dict(facecolor='black', shrink=0.05, width=2, headwidth=8))

ax.axis('off')
ax.set_title("Forward Pass Shape Flow", fontsize=16)
plt.show()

## 🎓 Exam Corner

### Question 1 (Practical)
Given input shape (32, 10), layer 1 has 16 neurons. What is the weight matrix shape? What is the output shape?

<details><summary>💡 Answer</summary>

**Weight matrix shape:** `(10, 16)` 
Because weights transform the input dimension (10) into the number of neurons (16).

**Output shape:** `(32, 16)`
Because the output contains one feature vector of size 16 for each of the 32 samples in the batch.
Calculation: `(32, 10) @ (10, 16) = (32, 16)`

</details>

### Question 2 (Conceptual)
If you remove the activation function from all hidden layers, what class of functions can the MLP represent?

<details><summary>💡 Answer</summary>

Only **linear functions**.

A series of linear transformations is mathematically equivalent to a single linear transformation.
$\mathbf{W_3}(\mathbf{W_2}(\mathbf{W_1}x)) = (\mathbf{W_3}\mathbf{W_2}\mathbf{W_1})x = \mathbf{W_{eq}}x$
Without non-linear activations (like ReLU or Sigmoid), an MLP with 100 hidden layers has exactly the same representational power as a 0-hidden-layer linear regression model.

</details>